## LOAD DATA

In [1]:
import pandas as pd
import numpy as np
from sklearn.preprocessing import MultiLabelBinarizer
import os
import re

In [2]:
# Charger le fichier
df_raw = pd.read_excel("raw_dataset.xlsx", header=None)

print(df_raw.shape)
df_raw.head(15)

(554, 37)


,0,1,2,3,4,5,6,7,8,9,...,27,28,29,30,31,32,33,34,35,36
0,R E P U B L I Q U E A L G E R I E N N E D ...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
1,MINISTERE DE L'AGRICULTURE ET DU DEVELOPPEMENT...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
2,CONSERVATION DES FORETS DE LA WILAYA DE: GUELMA,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
3,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
4,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
5,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
6,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,CAMPAGNES 2010 - 2025,NaN,NaN,NaN,NaN,NaN
7,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,...,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN,NaN
8,N°,DAIRA,COMMUNE,NOM FORET,COORDONNEES GEOGRAPHIQUES (UTM),NaN,NaN,DECLARATION,NaN,INTERVENTION,...,NaN,NaN,NaN,NaN,NaN,NaN,CONDITIONS METEO,ORGANISMES,EVALUATION,ENQUETE
9,NaN,NaN,NaN,OU,LONGITUDE,LATITUDE,NOM,DATE,HEURE,DATE,...,NaN,NaN,NaN,CAUSE,AUTEUR,SIGNALE,(TEMPERATURE ET VENTS),PARTICIPANTS,DES DEGATS,(*)


In [3]:
df1 = df_raw.iloc[11:].reset_index(drop=True)
df1.columns = [
    "ID", "DAIRA", "COMMUNE", "FORET",
    "LONGITUDE", "LATITUDE", "CART_NAME",
    
    "DATE_DECL", "HEURE_DECL",
    "DATE_INT", "HEURE_INT",
    "DATE_EXT", "HEURE_EXT",
    
    "ESSENCE",
    
    "DOM_FORET", "DOM_MAQUIS", "DOM_BROUSSAILLES", "DOM_ALFA", "DOM_AUTRES",
    "PRIV_FORET", "PRIV_MAQUIS", "PRIV_BROUSSAILLES", "PRIV_ALFA", "PRIV_AUTRES",
    
    "TOT_FORET", "TOT_MAQUIS", "TOT_BROUSSAILLES", "TOT_ALFA", "TOT_AUTRES",
    "SURF_TOTAL",
    
    "CAUSE", "AUTEUR", "SIGNALE",
    "METEO", "ORGANISMES",
    "DEGATS", 'ENQUETE'
]

df1 = df1.dropna(how="all").reset_index(drop=True)
df1

,ID,DAIRA,COMMUNE,FORET,LONGITUDE,LATITUDE,CART_NAME,DATE_DECL,HEURE_DECL,DATE_INT,...,TOT_ALFA,TOT_AUTRES,SURF_TOTAL,CAUSE,AUTEUR,SIGNALE,METEO,ORGANISMES,DEGATS,ENQUETE
0,01.24.28.10,HAMMAM DEBAGH,ROKNIA,DJ DEBAGH,NaN,NaN,NaN,2010-07-02 00:00:00,10:20:00,2010-07-02 00:00:00,...,-,-,2,INC,INCON,PV,NaN,SF + PC + DW,400000,ENGAGEE
1,02.24.16.10,GUELAAT BOU SBAA,BENI MEZLINE,KEF AMAR,NaN,NaN,NaN,2010-07-09 00:00:00,11:10:00,2010-07-09 00:00:00,...,-,-,1,INC,INCON,PV,NaN,SF + PC + DW,200000,ENGAGEE
2,03.24.33.10,HAMMAM N'BAILS,OUED CHEHAM,SAFIET AOUAYED,NaN,NaN,NaN,2010-07-12 00:00:00,16:25:00,2010-07-12 00:00:00,...,-,-,0.5,INC,INCON,BM,NaN,SF,50000,ENGAGEE
3,04.24.33.10,HAMMAM N'BAILS,OUED CHEHAM,OUE GHANEM,NaN,NaN,NaN,2010-07-13 00:00:00,11:45:00,2010-07-13 00:00:00,...,-,-,2.5,INC,INCON,BM,NaN,SF,250000,ENGAGEE
4,05.24.20.10,HELIOPOLIS,EL FEDJOUDJ,AIN TOUTA,NaN,NaN,NaN,2010-07-14 00:00:00,12:30:00,2010-07-14 00:00:00,...,-,-,2,INC,INCON,PV,NaN,SF,200000,ENGAGEE
...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...,...
538,11.24.22.25,HAMMAM N'BAIL,HAMMAM N'BAILS,SOUIBINA,7.629727°,36.238759°,GOOGLE EARTH,2025-08-27 00:00:00,19:55:00,2025-08-27 00:00:00,...,NaN,NaN,15,INC,INCON,PV,-,SF/PC/DW/EAPC/ANP/EMPS/VOL,NaN,ENGAGEE
539,12.24.30.25,BOUCHEGOUF,MEDJEZ SFA,F D BENI SALAH (LOURIDA),7.916214°,36.442024°,GOOGLE EARTH,2025-08-31 00:00:00,14:15:00,2025-08-31 00:00:00,...,NaN,NaN,65,INC,INCON,PV,-,SF/PC/DW/EAPC/SN/ANP/EMPS/VOL,NaN,ENGAGEE
540,13.24.09.25,HAMMAM N'BAIL,DAHOUARA,DJ AOURES,7.679589°,36.271054°,GOOGLE EARTH,2025-09-03 00:00:00,15:08:00,2025-09-03 00:00:00,...,NaN,NaN,0.5,INC,INCON,PV,-,SF/PC/EAPC/EMPS /DW,NaN,ENGAGEE
541,14.24.26.25,HELIOPOLIS,HELIOPOLIS,MECHTAT AIN RIHANA,7.399583°,36.561760°,GOOGLE EARTH,2025-09-07 00:00:00,14:45:00,2025-09-07 00:00:00,...,-,-,1.5,INC,INCON,PV,-,SF/PC/EAPC/ DW/EMPS,100000,ENGAGEE


## Clean Data

In [4]:
df2 = df1.replace(["", " ", "NaN", "nan", "NULL", "null", "None", '-'], np.nan)
nan_percent = (df2.isna().mean() * 100).sort_values(ascending=False)
print(nan_percent)

PRIV_ALFA            100.000000
PRIV_FORET           100.000000
PRIV_BROUSSAILLES     99.815838
DOM_AUTRES            99.631676
TOT_ALFA              99.631676
DOM_ALFA              99.631676
PRIV_MAQUIS           99.447514
TOT_AUTRES            99.447514
PRIV_AUTRES           94.290976
TOT_BROUSSAILLES      70.718232
DOM_BROUSSAILLES      68.692449
TOT_FORET             63.904236
DOM_FORET             61.325967
CART_NAME             57.826888
LONGITUDE             55.616943
LATITUDE              55.248619
TOT_MAQUIS            52.854512
DOM_MAQUIS            52.117864
METEO                 48.987109
DEGATS                 0.736648
ESSENCE                0.736648
HEURE_EXT              0.184162
SURF_TOTAL             0.000000
ORGANISMES             0.000000
SIGNALE                0.000000
AUTEUR                 0.000000
CAUSE                  0.000000
ID                     0.000000
DAIRA                  0.000000
DATE_EXT               0.000000
HEURE_INT              0.000000
DATE_INT

C:\Users\GEEK\AppData\Local\Temp\ipykernel_3792\250835585.py:1: FutureWarning: Downcasting behavior in `replace` is deprecated and will be removed in a future version. To retain the old behavior, explicitly call `result.infer_objects(copy=False)`. To opt-in to the future behavior, set `pd.set_option('future.no_silent_downcasting', True)`
  df2 = df1.replace(["", " ", "NaN", "nan", "NULL", "null", "None", '-'], np.nan)


In [5]:
#drop columns with more than 90% missing values + id + duplicated columns
df3  = df2.dropna(thresh=len(df2)*0.1, axis=1)
df3 = df3.drop(columns=['ID', 'DOM_FORET', 'DOM_MAQUIS', 'DOM_BROUSSAILLES'])

In [6]:
df4 = df3.copy()
# make TOT cols missing to 0
tot_cols = ['TOT_FORET', 'TOT_MAQUIS', 'TOT_BROUSSAILLES']
for col in tot_cols:
    df4[col] = df4[col].fillna(0)

In [7]:
df5 = df4.copy()

# Convert to datetime first
date_cols = ['DATE_DECL', 'DATE_INT', 'DATE_EXT']
time_cols = ['HEURE_DECL', 'HEURE_INT', 'HEURE_EXT']

for col in date_cols:
    df5[col] = pd.to_datetime(df5[col], errors='coerce')

for col in time_cols:
    df5[col] = pd.to_datetime(df5[col], format='%H:%M:%S', errors='coerce')

# Date features
df5['DECL_YEAR']  = df5['DATE_DECL'].dt.year
df5['DECL_MONTH'] = df5['DATE_DECL'].dt.month
df5['DECL_DAY']   = df5['DATE_DECL'].dt.day

df5['INT_YEAR']   = df5['DATE_INT'].dt.year
df5['INT_MONTH']  = df5['DATE_INT'].dt.month
df5['INT_DAY']    = df5['DATE_INT'].dt.day

df5['EXT_YEAR']   = df5['DATE_EXT'].dt.year
df5['EXT_MONTH']  = df5['DATE_EXT'].dt.month
df5['EXT_DAY']    = df5['DATE_EXT'].dt.day

# Time features
df5['DECL_HOUR'] = df5['HEURE_DECL'].dt.hour
df5['INT_HOUR']  = df5['HEURE_INT'].dt.hour
df5['EXT_HOUR']  = df5['HEURE_EXT'].dt.hour
df5.drop(columns=date_cols + time_cols, inplace=True)

In [8]:
#clean categorial features
df6 = df5.copy()
#clean daira
df6['DAIRA'] = df6['DAIRA'].replace('KHEZARAS', 'KHEZARA')
df6['DAIRA'] = df6['DAIRA'].replace("HAMMAM N'BAILS", "HAMMAM N'BAIL")
#clean forest
#replace any DJEBL in any part of the string with DJ
#strip leading and trailing spaces
df6['FORET'] = df6['FORET'].str.replace(r'\s+', ' ', regex=True).str.strip()
df6['FORET'] = df6['FORET'].str.replace(r'DJEBL|DJBEL|DJEBEL', 'DJ', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'ASEKRE|LASKRE|ASKAR', 'ASKAR', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'BOUGHERISSINE|BOUGHRESSINE', 'BOUGHRESINE', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'MEKHALHA', 'MEKHALFA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'LAZREG', 'LEZREG', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'MARMOURA|MERMOURA', 'MERMOURA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'ESSGHIRA|SAGHIRA', 'SEGHIRA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'MERIHIL', 'MRIHL', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'MUNCHAR', 'MUNCHORA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'FAROUDJA|FERODJA|FEROUDJA|FROUJA', 'FAROUJA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'CHAIR', 'CHEAIR', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'GUFAZA', 'GUEFAZA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'HMAM', 'HAMMAM', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'KORATH', 'KOURATH', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'KAF|KEF EL', 'KEF', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'SERRAK|SAREK', 'SERAK', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'LEGREIN', 'LEGRINE', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'LOUREIDA', 'LOURIDA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'Mt', 'MT', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'OUE', 'OUED', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'MEDJNOUNA', 'MEJNOUNA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'MEZAIR', 'MEZIAR', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'NASR', 'NASSER', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'SIDI REZIGU', 'SIDI REZIK', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'TOUIFEZA', 'TOUIFZA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'ZÉZOUA', 'ZIZOUA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'SAFI|SAFIAT', 'SAFIET', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'MACHTET LEGULALA', 'LEGULALA', regex=True)
df6['FORET'] = df6['FORET'].str.replace(r'SIDI ALI EL ARIANE|SIDI ALI LAARIANE', 'SIDI ALI ARIANE', regex=True)

In [9]:
df6['SIGNALE'] = df6['SIGNALE'].str.replace(r'CIT', 'CT', regex=True)

In [10]:
df6['ESSENCE'] = df6['ESSENCE'].astype('string')

# -------------------------
# 1. basic normalization
# -------------------------
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\s+', ' ', regex=True).str.strip()

# -------------------------
# 2. unify separators
# -------------------------
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\s*\+\s*', '+', regex=True)
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\s*-\s*', '+', regex=True)

# -------------------------
# 3. standardize labels
# -------------------------
df6['ESSENCE'] = df6['ESSENCE'].str.replace("CL (DEGR)", "CL", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("BROUSS", "BRS", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("BROSS", "BRS", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("CYPRE", "CYP", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("MAQUIS", "MAQ", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("T.PARCOUR", "TPARCOUR", regex=False)
df6['ESSENCE'] = df6['ESSENCE'].str.replace("تشجير", "REB", regex=False)

# -------------------------
# 4. clean bad separators
# -------------------------
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\++', '+', regex=True)   # remove multiple +
df6['ESSENCE'] = df6['ESSENCE'].str.strip('+ ')                        # trim edge +

# extract (REB) or any parenthesis content
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\(([^)]+)\)', r'+\1', regex=True)

# clean multiple +
df6['ESSENCE'] = df6['ESSENCE'].str.replace(r'\++', '+', regex=True)

# clean edges
df6['ESSENCE'] = df6['ESSENCE'].str.strip('+ ')

# -------------------------
# 5. final cleanup
# -------------------------
df6['ESSENCE'] = df6['ESSENCE'].replace(['', ' ', 'nan'], pd.NA)

In [11]:
df7 = df6.copy()
df7[['METEO_TEMP', 'METEO_VENTE_VITESSE', 'METEO_VENTE_DIRECTION']] = \
df7['METEO'].str.split(' - ', expand=True)

# 2. clean temperature (remove ° and convert to float)
df7['METEO_TEMP'] = (
    df7['METEO_TEMP']
    .str.replace('°', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# 3. clean wind speed (remove Km/h and convert to float)
df7['METEO_VENTE_VITESSE'] = (
    df7['METEO_VENTE_VITESSE']
    .str.replace('Km/h', '', regex=False)
    .str.replace(' ', '', regex=False)
    .str.replace(',', '.', regex=False)
    .astype(float)
)

# 4. clean direction
df7['METEO_VENTE_DIRECTION'] = (
    df7['METEO_VENTE_DIRECTION']
    .str.upper()
    .str.strip()
)
# 5. replace the direction full word with symbol
direction_map = {
    'NORD': 'N',
    'SUD': 'S',
    'EST': 'E',
    'OUEST': 'O',
    'NORD-EST': 'NE',
    'NORD-OUEST': 'NO',
    'SUD-EST': 'SE',
    'SUD-OUEST': 'SO'
}

df7['METEO_VENTE_DIRECTION'] = df7['METEO_VENTE_DIRECTION'].replace(direction_map)
df7['METEO_VENTE_DIRECTION'] = df7['METEO_VENTE_DIRECTION'].fillna('').str.replace('/', '+')

df7.drop(columns=['METEO'], inplace=True)
df7.head()

,DAIRA,COMMUNE,FORET,LONGITUDE,LATITUDE,CART_NAME,ESSENCE,TOT_FORET,TOT_MAQUIS,TOT_BROUSSAILLES,...,INT_DAY,EXT_YEAR,EXT_MONTH,EXT_DAY,DECL_HOUR,INT_HOUR,EXT_HOUR,METEO_TEMP,METEO_VENTE_VITESSE,METEO_VENTE_DIRECTION
0,HAMMAM DEBAGH,ROKNIA,DJ DEBAGH,NaN,NaN,NaN,MAQ,0.0,2.0,0.0,...,2,2010,7,2,10.0,10,15.0,NaN,NaN,
1,GUELAAT BOU SBAA,BENI MEZLINE,KEF AMAR,NaN,NaN,NaN,MAQ,0.0,1.0,0.0,...,9,2010,7,9,11.0,11,15.0,NaN,NaN,
2,HAMMAM N'BAIL,OUED CHEHAM,SAFIETET AOUAYED,NaN,NaN,NaN,BRS,0.0,0.0,0.5,...,12,2010,7,12,16.0,16,17.0,NaN,NaN,
3,HAMMAM N'BAIL,OUED CHEHAM,OUED GHANEM,NaN,NaN,NaN,BRS,0.0,0.0,2.5,...,13,2010,7,13,11.0,11,15.0,NaN,NaN,
4,HELIOPOLIS,EL FEDJOUDJ,AIN TOUTA,NaN,NaN,NaN,BRS,0.0,0.0,2.0,...,14,2010,7,14,12.0,12,15.0,NaN,NaN,


In [12]:
df8 = df7.copy()
def extract_number(x):
    if pd.isna(x):
        return np.nan

    x = str(x)

    # replace comma decimal → dot
    x = x.replace(',', '.')

    # remove spaces
    x = x.strip()

    # remove ranges → take first value
    if '-' in x:
        x = x.split('-')[0]

    # extract first number found
    match = re.search(r'[-+]?\d*\.?\d+', x)
    if match:
        return float(match.group())

    return np.nan

def dms_to_dd(x):
    if pd.isna(x):
        return np.nan

    x = str(x)

    pattern = r'(\d+)[°\s]+(\d+)[\'\s]+([\d\.]+)"?\s*([NSEW])?'
    m = re.search(pattern, x)

    if not m:
        return np.nan

    deg, minutes, seconds, direction = m.groups()

    dd = float(deg) + float(minutes)/60 + float(seconds)/3600

    if direction in ['S', 'W']:
        dd = -dd

    return dd

def clean_coord(x):
    if pd.isna(x):
        return np.nan

    x = str(x)

    # case 1: DMS format
    if "°" in x:
        return dms_to_dd(x)

    # case 2: numeric / UTM / range
    return extract_number(x)

df8['LONGITUDE'] = df8['LONGITUDE'].apply(clean_coord)
df8['LATITUDE']  = df8['LATITUDE'].apply(clean_coord)

df8['LOCATION_NAME'] = (
    df8['CART_NAME']
    .astype(str)
    .str.replace(r'\s+', ' ', regex=True)
    .str.strip()
)

df8.drop(columns=['CART_NAME'], inplace=True)


In [13]:
df9 = df8.copy()
df9.drop(columns=['LOCATION_NAME', "LONGITUDE", 'LATITUDE'], inplace=True)

## Feature engennering

### THEME 1 : Probality of happeinging in this zone

In [14]:
# show how many forest count less than 10 occurrences
counts = df9['FORET'].value_counts()
rare_forets = counts[(counts > 5)]
print(rare_forets)
print(f"Number: {len(rare_forets)}")

FORET
KEF SERAK         20
FAROUJA           11
OUEDD GHANEM      10
AIN EL OUAHCH      7
OULED CHIHA        7
DJ BOUGHRESINE     7
MOUBIA             7
KEF KOURATH        6
SEBAA MEZIAR       6
BENI SALAH         6
Name: count, dtype: int64
Number: 10


In [15]:
# remove unnecessary columns
df10 = df9.copy()
unc_cols = ['CAUSE', 'AUTEUR', 'SIGNALE', 'ORGANISMES', 'DEGATS', 'ENQUETE',
'INT_YEAR', 'INT_MONTH', 'INT_DAY', 'EXT_YEAR', 'EXT_MONTH', 'EXT_DAY',
'INT_HOUR', 'EXT_HOUR', 'DECL_HOUR']
df10 = df10.drop(columns=unc_cols)
#rename decl-times to just times
df10 = df10.rename(columns={'DECL_YEAR': 'YEAR', 'DECL_MONTH': 'MONTH', 'DECL_DAY': 'DAY'})


In [16]:
df11 = df10.copy()
df11['DATE'] = pd.to_datetime(df11[['YEAR','MONTH','DAY']])

# week features
df11['WEEK'] = df11['DATE'].dt.isocalendar().week
df11['WEEK_YEAR'] = df11['DATE'].dt.strftime('%Y-%W')

# season feature
def get_season(month):
    if month in [3, 4, 5]:
        return 'SPRING'
    elif month in [6, 7, 8]:
        return 'SUMMER'
    elif month in [9, 10, 11]:
        return 'AUTUMN'
    else:
        return 'WINTER'

df11['SEASON'] = df11['MONTH'].apply(get_season)
df11.head()



,DAIRA,COMMUNE,FORET,ESSENCE,TOT_FORET,TOT_MAQUIS,TOT_BROUSSAILLES,SURF_TOTAL,YEAR,MONTH,DAY,METEO_TEMP,METEO_VENTE_VITESSE,METEO_VENTE_DIRECTION,DATE,WEEK,WEEK_YEAR,SEASON
0,HAMMAM DEBAGH,ROKNIA,DJ DEBAGH,MAQ,0.0,2.0,0.0,2.0,2010,7,2,NaN,NaN,,2010-07-02,26,2010-26,SUMMER
1,GUELAAT BOU SBAA,BENI MEZLINE,KEF AMAR,MAQ,0.0,1.0,0.0,1.0,2010,7,9,NaN,NaN,,2010-07-09,27,2010-27,SUMMER
2,HAMMAM N'BAIL,OUED CHEHAM,SAFIETET AOUAYED,BRS,0.0,0.0,0.5,0.5,2010,7,12,NaN,NaN,,2010-07-12,28,2010-28,SUMMER
3,HAMMAM N'BAIL,OUED CHEHAM,OUED GHANEM,BRS,0.0,0.0,2.5,2.5,2010,7,13,NaN,NaN,,2010-07-13,28,2010-28,SUMMER
4,HELIOPOLIS,EL FEDJOUDJ,AIN TOUTA,BRS,0.0,0.0,2.0,2.0,2010,7,14,NaN,NaN,,2010-07-14,28,2010-28,SUMMER


In [17]:
#add occurances by daira and by 
df12 = df11.copy()
df12['COMMUNE_OCC'] = df12['COMMUNE'].map(df12['COMMUNE'].value_counts(normalize=True))
df12['DAIRA_OCC'] = df12['DAIRA'].map(df12['DAIRA'].value_counts(normalize=True))


In [18]:
df13 = df12.copy()
# count occurrences of each forest
foret_counts = df13['FORET'].value_counts()

# keep forests with more than 5 occurrences
valid_forets = foret_counts[foret_counts > 5].index

# create new grouped column
df13['FORET_GROUPED'] = df13['FORET'].apply(
    lambda x: x if x in valid_forets else 'OTHER'
)

In [19]:
df14 = df13.copy()
df14['FORET_CON'] = df14['TOT_FORET'] / df14['SURF_TOTAL']
df14['MAQUIS_CON'] = df14['TOT_MAQUIS'] / df14['SURF_TOTAL']
df14['BROUSSAILLES_CON'] = df14['TOT_BROUSSAILLES'] / df14['SURF_TOTAL']

In [20]:
df15 = df14.copy()

# -----------------------------
# 0. Clean categorical wind direction first
# -----------------------------
df15['METEO_VENTE_DIRECTION'] = (
    df15['METEO_VENTE_DIRECTION']
    .astype(str)
    .str.upper()
    .str.strip()
    .replace(['NAN', 'NONE', 'NULL', '', ' '], np.nan)
)

# -----------------------------
# 1. Ensure correct numeric types
# -----------------------------
df15['METEO_TEMP'] = pd.to_numeric(df15['METEO_TEMP'], errors='coerce')
df15['METEO_VENTE_VITESSE'] = pd.to_numeric(df15['METEO_VENTE_VITESSE'], errors='coerce')


# -----------------------------
# 2. Fill Temperature
# -----------------------------
df15['METEO_TEMP'] = df15['METEO_TEMP'].fillna(
    df15.groupby(['COMMUNE', 'MONTH'])['METEO_TEMP'].transform('median')
)

df15['METEO_TEMP'] = df15['METEO_TEMP'].fillna(df15['METEO_TEMP'].median())


# -----------------------------
# 3. Fill Wind Speed
# -----------------------------
df15['METEO_VENTE_VITESSE'] = df15['METEO_VENTE_VITESSE'].fillna(
    df15.groupby(['COMMUNE', 'MONTH'])['METEO_VENTE_VITESSE'].transform('median')
)

df15['METEO_VENTE_VITESSE'] = df15['METEO_VENTE_VITESSE'].fillna(
    df15['METEO_VENTE_VITESSE'].median()
)


# -----------------------------
# 4. Safe mode function (important fix)
# -----------------------------
def safe_mode(x):
    m = x.mode()
    if len(m) > 0:
        return m.iloc[0]
    return np.nan


# -----------------------------
# 5. Fill Wind Direction (by COMMUNE)
# -----------------------------
df15['METEO_VENTE_DIRECTION'] = df15['METEO_VENTE_DIRECTION'].fillna(
    df15.groupby('COMMUNE')['METEO_VENTE_DIRECTION'].transform(safe_mode)
)


# -----------------------------
# 6. Global fallback mode
# -----------------------------
global_mode = df15['METEO_VENTE_DIRECTION'].mode()
if len(global_mode) > 0:
    df15['METEO_VENTE_DIRECTION'] = df15['METEO_VENTE_DIRECTION'].fillna(global_mode[0])


# -----------------------------
# 7. Final safety fallback
# -----------------------------
df15['METEO_VENTE_DIRECTION'] = df15['METEO_VENTE_DIRECTION'].fillna('UNKNOWN')


# -----------------------------
# 8. Quick check
# -----------------------------
df15[['METEO_TEMP', 'METEO_VENTE_VITESSE', 'METEO_VENTE_DIRECTION', 'SEASON']].head()

,METEO_TEMP,METEO_VENTE_VITESSE,METEO_VENTE_DIRECTION,SEASON
0,44.0,28.0,S+SE,SUMMER
1,44.0,32.0,S+O,SUMMER
2,42.0,24.0,S+SE,SUMMER
3,42.0,24.0,S+SE,SUMMER
4,40.0,22.2,S+E,SUMMER


### Fire DF Building

In [21]:
reg_static_features = ['FORET', 'COMMUNE', 'DAIRA', 'ESSENCE', 'TOT_FORET', 'TOT_MAQUIS', 'TOT_BROUSSAILLES',
'SURF_TOTAL', 'COMMUNE_OCC', 'DAIRA_OCC', 'FORET_GROUPED', 'FORET_CON', 'MAQUIS_CON',
'BROUSSAILLES_CON']

time_static_features = ['FORET', 'DATE', 'MONTH', 'SEASON']
meteo_cols = ['METEO_TEMP', 'METEO_VENTE_VITESSE', 'METEO_VENTE_DIRECTION']


In [22]:
all_time = pd.MultiIndex.from_product(
    [df15['YEAR'].unique(), df15['WEEK'].unique()],
    names=['YEAR','WEEK']
).to_frame(index=False)

df_meteo = df15.groupby(['YEAR','WEEK']).agg({
    'METEO_TEMP': 'mean',
    'METEO_VENTE_VITESSE': 'mean',
    'METEO_VENTE_DIRECTION': lambda x: x.mode().iloc[0] if not x.mode().empty else np.nan
}).reset_index()

df_meteo = all_time.merge(df_meteo, on=['YEAR','WEEK'], how='left')

df_static = df15[reg_static_features].drop_duplicates(subset=['FORET'])
df_season = df15[['YEAR','WEEK','SEASON']].drop_duplicates()

In [23]:
import pandas as pd

# =========================
# 1. BUILD GRID (ALL POSSIBLE COMBINATIONS)
# =========================
forets = df15['FORET'].unique()
time = df15[['YEAR', 'WEEK']].drop_duplicates()

grid = pd.MultiIndex.from_product(
    [forets, time['YEAR'].unique(), time['WEEK'].unique()],
    names=['FORET', 'YEAR', 'WEEK']
).to_frame(index=False)

# =========================
# 2. FIRE LABEL (POSITIVE CLASS ONLY)
# =========================
fire = df15[['FORET', 'YEAR', 'WEEK']].drop_duplicates()
fire['FIRE_LABEL'] = 1

# =========================
# 3. MERGE FEATURES
# =========================
df_fire = grid.copy()

df_fire = df_fire.merge(df_static, on='FORET', how='left')
df_fire = df_fire.merge(df_meteo, on=['YEAR', 'WEEK'], how='left')
df_fire = df_fire.merge(df_season, on=['YEAR', 'WEEK'], how='left')
df_fire = df_fire.merge(fire, on=['FORET', 'YEAR', 'WEEK'], how='left')

# =========================
# 4. REMOVE ROWS WITH MISSING KEY FEATURES
# =========================
meteo_cols = ['METEO_TEMP', 'METEO_VENTE_VITESSE', 'METEO_VENTE_DIRECTION']

df_fire = df_fire.dropna(subset=meteo_cols)

# =========================
# 5. CREATE FINAL LABEL
# =========================
df_fire['FIRE_LABEL'] = df_fire['FIRE_LABEL'].fillna(0).astype(int)

# =========================
# 6. BALANCE DATA (REDUCE 0 CLASS)
# =========================
df_1 = df_fire[df_fire['FIRE_LABEL'] == 1]
df_0 = df_fire[df_fire['FIRE_LABEL'] == 0]

# keep only 3x negatives vs positives
ratio = 3

df_0_sampled = df_0.sample(
    n=min(len(df_0), len(df_1) * ratio),
    random_state=42
)

df_fire_balanced = pd.concat([df_1, df_0_sampled]).sample(
    frac=1,
    random_state=42
).reset_index(drop=True)

# =========================
# 7. FINAL CHECK
# =========================
print(df_fire_balanced['FIRE_LABEL'].value_counts())
print(df_fire_balanced.shape)

FIRE_LABEL
0    1578
1     526
Name: count, dtype: int64
(2104, 21)


In [24]:
cols_to_ignore = ['FORET']
df_fire_balanced = df_fire_balanced.drop(columns=cols_to_ignore)

In [25]:
df16 = df_fire_balanced.copy()

# All categorical features
cat_cols = [
    'FORET_GROUPED',
    'COMMUNE',
    'DAIRA',
    'ESSENCE',
    'METEO_VENTE_DIRECTION',
    'SEASON'
]

all_encoded = []

for col in cat_cols:
    
    # 1. Convert to safe multi-label format
    split_col = df16[col].fillna("NONE").astype(str).apply(
        lambda x: [i.strip() for i in x.split('+') if i.strip() not in ["OTHER", "NONE", ""]]
    )
    
    # 2. Fit MultiLabelBinarizer
    mlb = MultiLabelBinarizer()
    encoded = mlb.fit_transform(split_col)
    
    # 3. Convert to DataFrame
    encoded_df = pd.DataFrame(
        encoded,
        columns=[f"{col}_{c}" for c in mlb.classes_],
        index=df16.index
    )
    
    all_encoded.append(encoded_df)

# 4. Drop original categorical columns
df16 = df16.drop(columns=cat_cols)

# 5. Concatenate all encoded features
df16 = pd.concat([df16] + all_encoded, axis=1)

In [26]:
import sys
import site

# add user site-packages to path
site.addsitedir(r"C:\Users\geek\AppData\Roaming\Python\Python312\site-packages")

from xgboost import XGBClassifier

In [27]:
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, classification_report, roc_auc_score
# =========================
# 1. DEFINE TARGET + FEATURES
# =========================
target = 'FIRE_LABEL'

X = df16.drop(columns=[target])
y = df16[target]

# =========================
# 2. TRAIN / TEST SPLIT
# =========================
X_train, X_test, y_train, y_test = train_test_split(
    X, y,
    test_size=0.2,
    random_state=42,
    stratify=y   # important for imbalance
)

# =========================
# 3. TRAIN XGBOOST
# =========================
model = XGBClassifier(
    n_estimators=300,
    max_depth=6,
    learning_rate=0.05,
    subsample=0.8,
    colsample_bytree=0.8,
    eval_metric='logloss',
    scale_pos_weight=(y_train == 0).sum() / (y_train == 1).sum(),
    random_state=42
)

model.fit(X_train, y_train)

# =========================
# 4. PREDICTIONS
# =========================
y_pred = model.predict(X_test)
y_proba = model.predict_proba(X_test)[:, 1]

# =========================
# 5. EVALUATION
# =========================
print("Accuracy:", accuracy_score(y_test, y_pred))
print("ROC-AUC:", roc_auc_score(y_test, y_proba))

print("\nClassification Report:\n")
print(classification_report(y_test, y_pred))

Accuracy: 0.684085510688836
ROC-AUC: 0.6569620253164556

Classification Report:

              precision    recall  f1-score   support

           0       0.80      0.77      0.78       316
           1       0.38      0.44      0.41       105

    accuracy                           0.68       421
   macro avg       0.59      0.60      0.60       421
weighted avg       0.70      0.68      0.69       421



In [28]:
df77 = df67.copy()
cols = []

for col in cols:
    df7[col] = df7[col].fillna('')
    split_values = df7[col].apply(lambda x: x.split('+') if x != '' else [])

    mlb = MultiLabelBinarizer()

    encoded = pd.DataFrame(
        mlb.fit_transform(split_values),
        columns=mlb.classes_,
        index=df7.index
    )

    # 4. merge with original dataframe
    df7 = pd.concat([df7, encoded], axis=1)

    df7.drop(columns=[col], inplace=True)

df7.head()    

NameError: name 'df67' is not defined

## Save to Excel

In [ ]:
# Enregistrer le DataFrame nettoyé dans un nouveau fichier Excel
df9.to_excel("clean_dataset.xlsx", index=False)

In [ ]:
# Enregistrer le DataFrame nettoyé dans un nouveau fichier Excel dans le dossier Cleaned Data
name = "clean_dataset_theme1.xlsx"
output_path = os.path.join("Cleaned Data", name)
df16.to_excel(output_path, index=False)

## Save Mapping

In [ ]:
categorical_col = "VENT"

# create folder if it doesn't exist
os.makedirs("mapping", exist_ok=True)

# get unique values
unique_cats = df6[categorical_col].dropna().astype(str).unique()
unique_cats = sorted(unique_cats)

# create mapping
mapping = {cat: idx for idx, cat in enumerate(unique_cats)}

# convert to DataFrame
mapping_df = pd.DataFrame(
    list(mapping.items()),
    columns=[categorical_col, f"{categorical_col}_CODE"]
)

# save inside folder
output_path = os.path.join("mapping", f"{categorical_col}_mapping.xlsx")
mapping_df.to_excel(output_path, index=False)

print("Saved to:", output_path)